# Proyecto QTranslate LIFIA — Quirk → OpenQASM 3.0 → Ecosistema objetivo

Este cuaderno de pruebas pretende ejecutar una etapa de verificación funcional inicial sobre las traducciones, para ello ejecuta las siguientes subetapas:

1. Descarga de librerías externas necesarias
2. Traduce circuitos de [Quirk](https://algassert.com/quirk) a OpenQASM 3.0.

    Carga de los algoritmos de prueba 15 a partir del JSON algorithms.json y traduccion a OpenQasm (12 preexistentes y 3 generados por nosotros, path /algorithms_qasm)

3. Generación de circuitos visuales en Quirk (path /circuits_quirk)
4. Generación de circuitos visuales en OpenQasm (path /circuits_qasm)


1. Descarga de las librerías necesarias para la funcionalidad


In [19]:
import sys
import subprocess
import importlib.util

print('Python:', sys.executable)

packages = {
    'qiskit': 'qiskit',
    'matplotlib': 'matplotlib',
    'qiskit_aer': 'qiskit-aer',
    'pylatexenc': 'pylatexenc',
    'qiskit_qasm3_import': 'qiskit-qasm3-import',
    'pylatexenc': 'pylatexenc'
}

detalle_info = ''
paquetes_instalados = 0

for module_name, pip_name in packages.items():
    if importlib.util.find_spec(module_name) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pip_name])
        estado = 'no existente, se instaló correctamente'
        paquetes_instalados += 1
    else:
        estado = 'existente previamente, no se instaló'

    info = f"{pip_name} - {estado}"
    detalle_info += (', ' if detalle_info else '') + info
    print(f'Paquete: {info}')

print(f'\nCantidad de paquetes instalados en este kernel: {paquetes_instalados}')
print(f'Paquetes requeridos para la funcionalidad, disponibles actualmente en el kernel.')


Python: c:\Users\PC\desarrollo\QTransLIFIA\.venv\Scripts\python.exe
Paquete: qiskit - existente previamente, no se instaló
Paquete: matplotlib - existente previamente, no se instaló
Paquete: qiskit-aer - existente previamente, no se instaló
Paquete: pylatexenc - existente previamente, no se instaló
Paquete: qiskit-qasm3-import - existente previamente, no se instaló

Cantidad de paquetes instalados en este kernel: 0
Paquetes requeridos para la funcionalidad, disponibles actualmente en el kernel.


### Generamos una tabla (MarkDown) por cada algoritmo de prueba que vayamos a ejecutar (path /algorithms.json)


In [ ]:
import json
from pathlib import Path
from IPython.display import display, Markdown
import os, sys
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))
from utils.qutils import parse_quirk_url, quirk_to_qasm, quirk_circuit_info, encode_quirk_url


def load_algorithms():
    candidates = [
        Path.cwd() / "algorithms.json",
        Path.cwd() / "notebooks" / "algorithms.json",
        Path.cwd().parent / "notebooks" / "algorithms.json",
    ]
    for path in candidates:
        if path.exists():
            with path.open("r", encoding="utf-8") as f:
                return json.load(f)
    raise FileNotFoundError("No se encontró el archivo algorithms.json")


ALGORITHMS = load_algorithms()

def describe_algorithms():
    total_items = len(ALGORITHMS)
    print(f"Se recorrieron {total_items} items en ALGORITHMS")
    md = '| # | Algoritmo | Qubits | Descripción |\n|---|---|---|---|\n'
    for k, v in ALGORITHMS.items():
        info = quirk_circuit_info(v['url'])
        nq = info['n_qubits']
        md += f'| {k.split(".")[0]} | **{k.split(". ")[1]}** | {nq} | {v["desc"]} |\n'
    display(Markdown(md))

describe_algorithms()


Se recorrieron 13 items en ALGORITHMS.


| # | Algoritmo | Qubits | Descripción |
|---|---|---|---|
| 1 | **Shor** | 4 | Algoritmo de factorización de Shor (4 qubits) |
| 2 | **Bernstein-Vazirani** | 4 | Algoritmo de Bernstein-Vazirani (4 qubits) |
| 3 | **Grover** | 2 | Algoritmo de búsqueda de Grover (2 qubits) |
| 4 | **Deutsch-Jozsa** | 4 | Algoritmo de Deutsch-Jozsa (4 qubits) |
| 5 | **Simon** | 6 | Algoritmo de Simon (6 qubits) |
| 6 | **TSP** | 3 | Circuito para problema del viajante (TSP, 3 qubits) |
| 7 | **Teleportation** | 5 | Protocolo de teleportación cuántica (5 qubits) |
| 8 | **Phase Estimation** | 4 | Estimación de fase cuántica (4 qubits + compuertas custom) |
| 9 | **QFT** | 3 | Transformada de Fourier Cuántica (3 qubits) |
| 10 | **QAOA** | 2 | Quantum Approximate Optimization Algorithm (2 qubits) |
| 11 | **Kickback** | 2 | Phase kickback (2 qubits) |
| 12 | **Full Adder** | 4 | Sumador completo (4 qubits) |
| 13 | **Multicontroled Gates** | 5 | Circuito de prueba con compuertas multicontroladas CCX, CCY, CCZ, CCCX, CCCY y CCCZ |


2. Traducción de algoritmos (algorithms.json) desde Quirk --> OpenQASM 3.0


### Generación de circuitos en formato textual (.txt) en OpenQASM3


In [17]:
import os
import pandas as pd
from pathlib import Path

# Creamos un directorio para almacenar los archivos QASM generados
output_dir = Path.cwd() / "algorithms_qasm"
output_dir.mkdir(exist_ok=True)


def generate_qasm_table_and_files():
    rows = []
    generated_paths = []
    for name, data in ALGORITHMS.items():
        url = data["url"]
        info = quirk_circuit_info(url)
        qasm_code = quirk_to_qasm(url, data.get("offset", 0))

        safe_name = name.replace(".", "_").replace(" ", "_")
        file_path = output_dir / f"{safe_name}.txt"
        file_path.write_text(qasm_code, encoding="utf-8")
        generated_paths.append(file_path.resolve())

        rows.append({
            "Algoritmo": name.split(". ", 1)[1] if ". " in name else name,
            "Qubits": info["n_qubits"],
            "Columnas": info["n_cols"],
            "Descripcion": data.get("desc", ""),
            "Archivo": str(file_path.name),
            "Líneas": len(qasm_code.splitlines())
        })

    total_archivos = len(rows)
    print(f"Todos los archivos generados en el path: /{output_dir.name} ({total_archivos} archivos)")
    for generated_path in generated_paths:
        print(f"- {generated_path.name}")

    df = pd.DataFrame(rows)
    return df


generate_qasm_table_and_files()


Todos los archivos generados en el path: /algorithms_qasm (13 archivos)
- 1__Shor.txt
- 2__Bernstein-Vazirani.txt
- 3__Grover.txt
- 4__Deutsch-Jozsa.txt
- 5__Simon.txt
- 6__TSP.txt
- 7__Teleportation.txt
- 8__Phase_Estimation.txt
- 9__QFT.txt
- 10__QAOA.txt
- 11__Kickback.txt
- 12__Full_Adder.txt
- 13__Multicontroled_Gates.txt


,Algoritmo,Qubits,Columnas,Descripcion,Archivo,Líneas
0,Shor,4,13,Algoritmo de factorización de Shor (4 qubits),1__Shor.txt,26
1,Bernstein-Vazirani,4,8,Algoritmo de Bernstein-Vazirani (4 qubits),2__Bernstein-Vazirani.txt,18
2,Grover,2,8,Algoritmo de búsqueda de Grover (2 qubits),3__Grover.txt,13
3,Deutsch-Jozsa,4,9,Algoritmo de Deutsch-Jozsa (4 qubits),4__Deutsch-Jozsa.txt,22
4,Simon,6,8,Algoritmo de Simon (6 qubits),5__Simon.txt,18
5,TSP,3,6,"Circuito para problema del viajante (TSP, 3 qu...",6__TSP.txt,13
6,Teleportation,5,8,Protocolo de teleportación cuántica (5 qubits),7__Teleportation.txt,13
7,Phase Estimation,4,17,Estimación de fase cuántica (4 qubits + compue...,8__Phase_Estimation.txt,26
8,QFT,3,8,Transformada de Fourier Cuántica (3 qubits),9__QFT.txt,15
9,QAOA,2,12,Quantum Approximate Optimization Algorithm (2 ...,10__QAOA.txt,8


### Generación de imágenes de circuitos originales Quirk

Genera archivos en el path: /circuits_quirk


In [35]:
from pathlib import Path

quirk_dir = Path.cwd() / "circuits_quirk"
quirk_dir.mkdir(exist_ok=True)

def render_quirk_circuits():
    for name, data in sorted(ALGORITHMS.items(), key=lambda item: [int(part) if part.isdigit() else part.lower() for part in item[0].split(".")[0].split("_")]):
        url = data["url"]
        safe_name = name.replace(".", "_").replace(" ", "_")
        png_path = quirk_dir / f"{safe_name}.png"

        try:
            qasm_code = quirk_to_qasm(url, data.get("offset", 0))
            circuit = load_qasm_circuit(qasm_code)
            circuit.draw(output='mpl', filename=str(png_path), style='bw')
            print(f"Generado: {png_path.name}")
        except Exception as exc:
            print(f"No se pudo generar Quirk {safe_name}: {exc}")

print(f"Generación de imágenes de circuitos originales Quirk ( {quirk_dir.resolve()} )\n")
render_quirk_circuits()
print(f"\n {len(ALGORITHMS)} Circuitos Quirk generados ( {quirk_dir.resolve()})")


Generación de imágenes de circuitos originales Quirk ( C:\Users\PC\desarrollo\QTransLIFIA\notebooks\circuits_quirk )

Generado: 1__Shor.png
Generado: 2__Bernstein-Vazirani.png
Generado: 3__Grover.png
Generado: 4__Deutsch-Jozsa.png
Generado: 5__Simon.png
Generado: 6__TSP.png
Generado: 7__Teleportation.png
Generado: 8__Phase_Estimation.png
Generado: 9__QFT.png
Generado: 10__QAOA.png
Generado: 11__Kickback.png
Generado: 12__Full_Adder.png
Generado: 13__Multicontroled_Gates.png

 13 Circuitos Quirk generados ( C:\Users\PC\desarrollo\QTransLIFIA\notebooks\circuits_quirk)


### Generación de imágenes de circuitos traducidos OpenQASM3

Genera archivos en el path: /circuits_qasm


In [43]:
from pathlib import Path
import re
from qiskit import QuantumCircuit
from qiskit.qasm3 import loads as qasm3_loads


circuits_dir = Path.cwd() / 'circuits_qasm'
circuits_dir.mkdir(exist_ok=True)


def natural_sort_key(value):
    parts = re.split(r'[_\s.-]+', str(value))
    return [int(p) if p.isdigit() else p.lower() for p in parts if p]


def load_qasm_circuit(qasm_text: str):
    text = (qasm_text or '').strip()
    if not text:
        raise ValueError('El contenido QASM está vacío')

    upper = text.upper()
    if upper.startswith('OPENQASM 3.0'):
        return qasm3_loads(text)
    if upper.startswith('OPENQASM 2.0'):
        return QuantumCircuit.from_qasm_str(text)

    try:
        return qasm3_loads(text)
    except Exception:
        return QuantumCircuit.from_qasm_str(text)


def render_qasm_circuits():
    qasm_dir = Path.cwd() / 'algorithms_qasm'
    circuits_dir = Path.cwd() / 'circuits_qasm'
    circuits_dir.mkdir(exist_ok=True)

    if not qasm_dir.exists():
        raise FileNotFoundError(f'No existe la carpeta de QASM: {qasm_dir}')

    files = sorted(qasm_dir.glob('*.txt'), key=lambda p: natural_sort_key(p.stem))
    if not files:
        raise FileNotFoundError(f'No hay archivos .txt en {qasm_dir}')

    for txt_path in files:
        qasm_text = txt_path.read_text(encoding='utf-8')

        try:
            circuit = load_qasm_circuit(qasm_text)
        except Exception as exc:
            print(f'No se pudo cargar {txt_path.name}: {exc}')
            continue

        png_path = circuits_dir / f'{txt_path.stem}.png'
        try:
            circuit.draw(output='mpl', filename=str(png_path), style='bw')
            print(f'Generado: {png_path.name}')
        except Exception as exc:
            print(f'No se pudo dibujar {txt_path.name}: {exc}')


print(f'Generación de imágenes de circuitos traducidos a OpenQASM 3.0 ( {circuits_dir.resolve()} )\n')
render_qasm_circuits()
print(f'\n {len(list((Path.cwd() / "algorithms_qasm").glob("*.txt")))} Circuitos OpenQASM 3.0 generados ( {circuits_dir.resolve()} )')


Generación de imágenes de circuitos traducidos a OpenQASM 3.0 ( C:\Users\PC\desarrollo\QTransLIFIA\notebooks\circuits_qasm )

Generado: 1__Shor.png
Generado: 2__Bernstein-Vazirani.png
Generado: 3__Grover.png
Generado: 4__Deutsch-Jozsa.png
Generado: 5__Simon.png
Generado: 6__TSP.png
Generado: 7__Teleportation.png
Generado: 8__Phase_Estimation.png
Generado: 9__QFT.png
Generado: 10__QAOA.png
Generado: 11__Kickback.png
Generado: 12__Full_Adder.png
Generado: 13__Multicontroled_Gates.png

 13 Circuitos OpenQASM 3.0 generados ( C:\Users\PC\desarrollo\QTransLIFIA\notebooks\circuits_qasm )


### Generamos una carpeta de comparación para agilizar la verificación de equivalencia visual
Carpeta /compare

In [44]:
from pathlib import Path
import re

project_root = Path.cwd()
compare_dir = project_root / 'compare'
compare_dir.mkdir(exist_ok=True)

qasm_dir = project_root / 'circuits_qasm'
quirk_dir = project_root / 'circuits_quirk'

if not qasm_dir.exists():
    raise FileNotFoundError(f'No existe la carpeta de circuitos OpenQASM: {qasm_dir}')
if not quirk_dir.exists():
    raise FileNotFoundError(f'No existe la carpeta de circuitos Quirk: {quirk_dir}')


def natural_sort_key(value):
    parts = re.split(r'[_\s.-]+', str(value))
    return [int(p) if p.isdigit() else p.lower() for p in parts if p]


qasm_pngs = {p.stem: p for p in sorted(qasm_dir.glob('*.png'), key=lambda p: natural_sort_key(p.stem))}
quirk_pngs = {p.stem: p for p in sorted(quirk_dir.glob('*.png'), key=lambda p: natural_sort_key(p.stem))}

all_names = sorted(set(qasm_pngs) | set(quirk_pngs), key=natural_sort_key)

index_lines = ['# Comparación de circuitos', '']

for stem in all_names:
    qasm_path = qasm_pngs.get(stem)
    quirk_path = quirk_pngs.get(stem)

    if qasm_path is None or quirk_path is None:
        continue

    title = stem.replace('_', ' ')
    md_path = compare_dir / f'{stem}.md'
    md_path.write_text(
        '\n'.join([
            f'# Comparación: {title}',
            '',
            '<table>',
            '<tr>',
            f"<td align='center'><b>OpenQASM 3.0</b><br><img src='../circuits_qasm/{qasm_path.name}' alt='{qasm_path.name}' width='420' /></td>",
            f"<td align='center'><b>Quirk</b><br><img src='../circuits_quirk/{quirk_path.name}' alt='{quirk_path.name}' width='420' /></td>",
            '</tr>',
            '</table>',
            '',
            f'Archivo QASM: `{qasm_path.name}`',
            f'Archivo Quirk: `{quirk_path.name}`',
        ]),
        encoding='utf-8'
    )

    index_lines.append(f'- [{title}]({stem}.md)')
    print(f'Generado comparativo: {md_path.name}')

index_path = compare_dir / 'README.md'
index_path.write_text('\n'.join(index_lines) + '\n', encoding='utf-8')

print(f'\nCarpeta de comparación creada en: /{compare_dir.name}')
print(f'Se generaron {len(all_names)} comparativas visuales.')
print(f'Índice disponible en: /{compare_dir.name}/README.md')


Generado comparativo: 1__Shor.md
Generado comparativo: 2__Bernstein-Vazirani.md
Generado comparativo: 3__Grover.md
Generado comparativo: 4__Deutsch-Jozsa.md
Generado comparativo: 5__Simon.md
Generado comparativo: 6__TSP.md
Generado comparativo: 7__Teleportation.md
Generado comparativo: 8__Phase_Estimation.md
Generado comparativo: 9__QFT.md
Generado comparativo: 10__QAOA.md
Generado comparativo: 11__Kickback.md
Generado comparativo: 12__Full_Adder.md
Generado comparativo: 13__Multicontroled_Gates.md

Carpeta de comparación creada en: /compare
Se generaron 13 comparativas visuales.
Índice disponible en: /compare/README.md


### Establecemos una comparación más robusta (semántica) mediante equivalencias de operadores

In [55]:
from pathlib import Path
import numpy as np
from qiskit import QuantumCircuit
from qiskit.qasm3 import loads as qasm3_loads
from qiskit.quantum_info import Operator


qasm_dir = Path.cwd() / 'algorithms_qasm'


def natural_sort_key(value):
    """Extrae el número al principio del nombre para ordenar naturalmente"""
    import re
    text = str(value).replace('.txt', '').strip()
    match = re.search(r'^(\d+)', text)
    if match:
        num = int(match.group(1))
        return (num, text)
    else:
        return (float('inf'), text)


def load_qasm_circuit(qasm_text: str):
    text = (qasm_text or '').strip()
    if not text:
        raise ValueError('El contenido QASM está vacío')
    upper = text.upper()
    if upper.startswith('OPENQASM 3.0'):
        return qasm3_loads(text)
    if upper.startswith('OPENQASM 2.0'):
        return QuantumCircuit.from_qasm_str(text)
    try:
        return qasm3_loads(text)
    except Exception:
        return QuantumCircuit.from_qasm_str(text)


def remove_measurements(circuit):
    """Elimina todas las mediciones y operaciones condicionales de un circuito"""
    from qiskit import QuantumCircuit
    clean = circuit.copy()
    
    # Remover mediciones finales
    clean.remove_final_measurements()
    
    # Remover operaciones condicionales (if_test) que dependan de mediciones
    instructions_to_remove = []
    for i, instr in enumerate(clean.data):
        if hasattr(instr.operation, 'condition') and instr.operation.condition is not None:
            instructions_to_remove.append(i)
        elif instr.operation.name == 'measure':
            instructions_to_remove.append(i)
    
    # Remover en orden inverso para no afectar índices
    for i in reversed(instructions_to_remove):
        clean.data.pop(i)
    
    return clean


def validate_qasm_equivalence():
    results = []
    for name, data in sorted(ALGORITHMS.items(), key=lambda item: natural_sort_key(item[0])):
        safe_name = name.replace('.', '_').replace(' ', '_')
        qasm_file = qasm_dir / f'{safe_name}.txt'
        if not qasm_file.exists():
            print(f'⊘ Salta {name}: no existe archivo QASM')
            continue

        try:
            qasm_text = qasm_file.read_text(encoding='utf-8')
            circuit_from_file = load_qasm_circuit(qasm_text)
            circuit_from_url = load_qasm_circuit(quirk_to_qasm(data['url'], data.get('offset', 0)))

            # Remover mediciones si existen
            circuit_from_file = remove_measurements(circuit_from_file)
            circuit_from_url = remove_measurements(circuit_from_url)

            u_file = Operator(circuit_from_file).data
            u_url = Operator(circuit_from_url).data
            max_diff = float(np.max(np.abs(u_file - u_url)))
            ok = max_diff < 1e-8

            results.append((name, ok, max_diff))
            if ok:
                status = '✅ EQUIVALENTE'
            else:
                status = '⚠️  NO EQUIVALENTE'
            print(f'{name}: {status} | max_diff = {max_diff:.3e}')
            
        except Exception as exc:
            print(f'{name}: ❌ ERROR -> {exc}')
            results.append((name, False, float('nan')))

    total_ok = sum(1 for _, ok, _ in results if ok)
    print(f'\n✓ Validación final: {total_ok}/{len(results)} circuitos equivalentes según comparación unitaria.')
    return results


results = validate_qasm_equivalence()


1. Shor: ✅ EQUIVALENTE | max_diff = 0.000e+00
2. Bernstein-Vazirani: ✅ EQUIVALENTE | max_diff = 0.000e+00


3. Grover: ✅ EQUIVALENTE | max_diff = 0.000e+00
4. Deutsch-Jozsa: ✅ EQUIVALENTE | max_diff = 0.000e+00
5. Simon: ✅ EQUIVALENTE | max_diff = 0.000e+00
6. TSP: ✅ EQUIVALENTE | max_diff = 0.000e+00
7. Teleportation: ✅ EQUIVALENTE | max_diff = 0.000e+00
8. Phase Estimation: ✅ EQUIVALENTE | max_diff = 0.000e+00
9. QFT: ✅ EQUIVALENTE | max_diff = 0.000e+00
10. QAOA: ✅ EQUIVALENTE | max_diff = 0.000e+00
11. Kickback: ✅ EQUIVALENTE | max_diff = 0.000e+00
12. Full Adder: ✅ EQUIVALENTE | max_diff = 0.000e+00
13. Multicontroled Gates: ✅ EQUIVALENTE | max_diff = 0.000e+00

✓ Validación final: 13/13 circuitos equivalentes según comparación unitaria.
